# Packages

In [29]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import random as rd
from surprise import AlgoBase
from surprise.prediction_algorithms.predictions import PredictionImpossible

from loaders import load_ratings
from loaders import load_items
from constants import Constant as C
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, RidgeCV, ElasticNet, ElasticNetCV
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Explore and select content features

In [30]:
df_items = load_items()
df_ratings = load_ratings()

# Example 1 : create title_length features
df_features = df_items[C.LABEL_COL].apply(lambda x: len(x)).to_frame('n_character_title')
display(df_features.head())

# (explore here other features)


,n_character_title
movieId,
1,16
2,14
3,23
4,24
5,34


# Build a content-based model
When ready, move the following class in the *models.py* script

In [34]:
class ContentBased(AlgoBase):
    def __init__(self, features_method, regressor_method, knn_k=30):
        AlgoBase.__init__(self)
        self.regressor_method = regressor_method
        self.knn_k = knn_k
        self.feature_groups = None
        self.content_features = self.create_content_features(features_method)

    # ── Helpers ───────────────────────────────────────────────────────────

    @staticmethod
    def _l2_normalize_rows(matrix):
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)
        return matrix / norms

    def _load_genome(self):
        df_genome = pd.read_csv(C.CONTENT_PATH / 'genome-scores.csv')
        df_genome = df_genome.pivot(index='movieId', columns='tagId', values='relevance')
        return df_genome.fillna(0)

    def _load_genome_scaled(self):
        from sklearn.preprocessing import StandardScaler
        df_genome = self._load_genome()
        scaler = StandardScaler()
        scaled = scaler.fit_transform(df_genome)
        return pd.DataFrame(scaled, index=df_genome.index, columns=df_genome.columns)

    def _load_genome_pruned(self, var_threshold=0.01):
        from sklearn.preprocessing import StandardScaler
        df_genome = self._load_genome()
        keep = df_genome.var(axis=0) > var_threshold
        df_genome = df_genome.loc[:, keep]
        scaler = StandardScaler()
        scaled = scaler.fit_transform(df_genome)
        return pd.DataFrame(scaled, index=df_genome.index,
                            columns=[f'g_{c}' for c in df_genome.columns])

    def _load_visuals_scaled(self):
        from sklearn.preprocessing import StandardScaler
        visual_path = C.CONTENT_PATH / 'visuals' / 'LLVisualFeatures13K_Log.csv'
        df_visuals = pd.read_csv(visual_path)
        df_visuals = df_visuals.set_index('ML_Id')
        df_visuals = df_visuals.fillna(df_visuals.mean())
        scaler = StandardScaler()
        scaled = scaler.fit_transform(df_visuals)
        return pd.DataFrame(scaled, index=df_visuals.index, columns=df_visuals.columns)

    def _load_tmdb_features(self,
                            n_lang=30, n_country=30, n_spoken=20,
                            n_studio=100, n_director=150, n_cast=500):
        """Load TMDB-derived features from local cache (run fetch_tmdb.py once)."""
        import json
        import re
        from collections import Counter

        cache_path = C.CONTENT_PATH / 'tmdb_cache.json'
        if not cache_path.exists():
            print(f"[TMDB] Cache not found at {cache_path}. Run 'python fetch_tmdb.py' first.")
            return None

        with open(cache_path) as f:
            cache = json.load(f)
        rows = [{'movieId': int(k), **v} for k, v in cache.items() if v is not None]
        if not rows:
            return None
        df = pd.DataFrame(rows).set_index('movieId')

        blocks = []
        runtime = pd.to_numeric(df['runtime'], errors='coerce')
        runtime = runtime.where(runtime > 0)
        runtime = runtime.fillna(runtime.mean())
        std = runtime.std() if runtime.std() > 0 else 1.0
        blocks.append(pd.DataFrame({'tmdb_runtime': (runtime - runtime.mean()) / std}))

        def safe(name):
            return re.sub(r'[^a-zA-Z0-9_]+', '_', str(name)).strip('_').lower()

        def multi_hot_topk(series, top_k, prefix):
            counts = Counter()
            for val in series:
                if isinstance(val, list):
                    counts.update(v for v in val if v)
                elif pd.notna(val) and val:
                    counts[val] += 1
            top_items = [it for it, _ in counts.most_common(top_k)]
            idx_map = {it: i for i, it in enumerate(top_items)}

            mat = np.zeros((len(series), len(top_items)), dtype=np.float32)
            for i, val in enumerate(series):
                if isinstance(val, list):
                    for v in val:
                        j = idx_map.get(v)
                        if j is not None:
                            mat[i, j] = 1.0
                elif pd.notna(val) and val:
                    j = idx_map.get(val)
                    if j is not None:
                        mat[i, j] = 1.0
            cols = [f'tmdb_{prefix}_{safe(it)}' for it in top_items]
            return pd.DataFrame(mat, index=series.index, columns=cols)

        blocks.append(multi_hot_topk(df['original_language'], n_lang, 'lang'))
        blocks.append(multi_hot_topk(df['production_countries'], n_country, 'country'))
        blocks.append(multi_hot_topk(df['spoken_languages'], n_spoken, 'spoken'))
        blocks.append(multi_hot_topk(df['production_companies'], n_studio, 'studio'))
        blocks.append(multi_hot_topk(df['directors'], n_director, 'director'))
        blocks.append(multi_hot_topk(df['cast_top5'], n_cast, 'cast'))

        result = pd.concat(blocks, axis=1)
        result = result[~result.index.duplicated(keep='first')]
        return result

    def _load_tags(self, max_features=200, ngrams=(1, 1), sublinear=False, min_df=2):
        df_tags = pd.read_csv(C.CONTENT_PATH / 'tags.csv')
        df_tags = df_tags.dropna(subset=['tag'])
        df_tags['tag'] = df_tags.tag.astype(str).str.lower()
        df_tags_grouped = df_tags.groupby('movieId')['tag'].apply(' '.join).reset_index().set_index('movieId')

        tfidf = TfidfVectorizer(
            max_features=max_features,
            min_df=min_df,
            ngram_range=ngrams,
            sublinear_tf=sublinear,
            stop_words='english'
        )
        tfidf_matrix = tfidf.fit_transform(df_tags_grouped['tag'])
        return pd.DataFrame(
            tfidf_matrix.toarray(),
            index=df_tags_grouped.index,
            columns=[f'tag_{c}' for c in tfidf.get_feature_names_out()]
        )

    def _load_year_genres(self, df_items, with_decades=False):
        years = df_items[C.LABEL_COL].str.extract(r'\((\d{4})\)').astype(float)
        years.columns = ['release_year']
        years = years.fillna(years.mean())
        df_year = (years - years.mean()) / years.std()

        extras = []
        if with_decades:
            bins = pd.cut(
                years['release_year'],
                bins=[0, 1970, 1980, 1990, 2000, 2010, 3000],
                labels=['era_pre70', 'era_70s', 'era_80s', 'era_90s', 'era_00s', 'era_10s']
            )
            df_decade = pd.get_dummies(bins, prefix='', prefix_sep='').astype(float)
            df_decade.index = df_items.index
            extras.append(df_decade)

            n_genres = df_items[C.GENRES_COL].fillna('').str.count(r'\|') + 1
            n_genres = (n_genres - n_genres.mean()) / n_genres.std()
            extras.append(pd.DataFrame({'n_genres': n_genres.values}, index=df_items.index))

        tfidf = TfidfVectorizer(token_pattern=r'[^|]+')
        tfidf_matrix = tfidf.fit_transform(df_items[C.GENRES_COL].fillna(''))
        df_genres = pd.DataFrame(
            tfidf_matrix.toarray(),
            index=df_items.index,
            columns=[f'genre_{c}' for c in tfidf.get_feature_names_out()]
        )
        if extras:
            df_year = pd.concat([df_year] + extras, axis=1)
        return df_year, df_genres

    @staticmethod
    def _scale_block(df, target_norm=1.0):
        values = df.values
        f_norm = np.linalg.norm(values, ord='fro')
        if f_norm < 1e-9:
            return df
        scale = target_norm * np.sqrt(values.shape[1]) / f_norm
        return df * scale

    # ── Content Analyzer ──────────────────────────────────────────────────

    def create_content_features(self, features_method):
        df_items = load_items()

        if features_method is None:
            return None

        elif features_method == "genome_scaled":
            return self._load_genome_scaled()

        elif features_method == "genome_scaled_tags":
            df_genome_scaled = self._load_genome_scaled()
            df_tags = self._load_tags()
            return df_genome_scaled.join(df_tags, how='left').fillna(0)

        elif features_method == "genome_scaled_visuals_scaled":
            df_genome_scaled = self._load_genome_scaled()
            df_visuals_scaled = self._load_visuals_scaled()
            return df_genome_scaled.join(df_visuals_scaled, how='left').fillna(0)

        elif features_method == "genome_scaled_tags_visuals_scaled":
            df_genome_scaled = self._load_genome_scaled()
            df_tags = self._load_tags()
            df_visuals_scaled = self._load_visuals_scaled()
            df_features = df_genome_scaled.join(df_tags, how='left')
            df_features = df_features.join(df_visuals_scaled, how='left')
            return df_features.fillna(0)

        elif features_method == "all_content":
            df_genome_scaled = self._load_genome_scaled()
            df_tags = self._load_tags()
            df_year, df_genres = self._load_year_genres(df_items)
            df_features = df_genome_scaled.join(df_tags, how='outer')
            df_features = df_features.join(df_year, how='outer')
            df_features = df_features.join(df_genres, how='outer')
            return df_features.fillna(0)

        elif features_method == "all_content_rich_tags":
            df_genome_scaled = self._load_genome_scaled()
            df_tags = self._load_tags(max_features=500, ngrams=(1, 2), sublinear=True, min_df=3)
            df_year, df_genres = self._load_year_genres(df_items)
            df_features = df_genome_scaled.join(df_tags, how='outer')
            df_features = df_features.join(df_year, how='outer')
            df_features = df_features.join(df_genres, how='outer')
            return df_features.fillna(0)

        elif features_method == "all_content_decade":
            df_genome_scaled = self._load_genome_scaled()
            df_tags = self._load_tags()
            df_year, df_genres = self._load_year_genres(df_items, with_decades=True)
            df_features = df_genome_scaled.join(df_tags, how='outer')
            df_features = df_features.join(df_year, how='outer')
            df_features = df_features.join(df_genres, how='outer')
            return df_features.fillna(0)

        elif features_method == "all_content_full":
            df_genome_scaled = self._load_genome_scaled()
            df_tags = self._load_tags(max_features=500, ngrams=(1, 2), sublinear=True, min_df=3)
            df_year, df_genres = self._load_year_genres(df_items, with_decades=True)
            df_features = df_genome_scaled.join(df_tags, how='outer')
            df_features = df_features.join(df_year, how='outer')
            df_features = df_features.join(df_genres, how='outer')
            return df_features.fillna(0)

        elif features_method == "all_content_tmdb":
            # all_content_full + TMDB metadata (runtime, lang, country, studio, director, cast)
            df_genome_scaled = self._load_genome_scaled()
            df_tags = self._load_tags(max_features=500, ngrams=(1, 2), sublinear=True, min_df=3)
            df_year, df_genres = self._load_year_genres(df_items, with_decades=True)
            df_tmdb = self._load_tmdb_features()
            df_features = df_genome_scaled.join(df_tags, how='outer')
            df_features = df_features.join(df_year, how='outer')
            df_features = df_features.join(df_genres, how='outer')
            if df_tmdb is not None:
                df_features = df_features.join(df_tmdb, how='outer')
            else:
                print("[all_content_tmdb] Warning: TMDB cache missing, falling back to all_content_full")
            return df_features.fillna(0)

        elif features_method == "all_content_v2":
            df_genome = self._load_genome_pruned(var_threshold=0.01)
            df_tags = self._load_tags(max_features=1000, ngrams=(1, 2), sublinear=True, min_df=3)
            df_year, df_genres = self._load_year_genres(df_items, with_decades=True)

            df_genome = self._scale_block(df_genome.fillna(0))
            df_tags = self._scale_block(df_tags.fillna(0))
            df_year = self._scale_block(df_year.fillna(0))
            df_genres = self._scale_block(df_genres.fillna(0))

            df_features = df_genome.join(df_tags, how='outer')
            df_features = df_features.join(df_year, how='outer')
            df_features = df_features.join(df_genres, how='outer')
            df_features = df_features.fillna(0)

            cols = list(df_features.columns)
            self.feature_groups = {
                'genome': [c for c in cols if c.startswith('g_')],
                'tags': [c for c in cols if c.startswith('tag_')],
                'year': [c for c in cols if c.startswith('release_year') or c.startswith('era_') or c == 'n_genres'],
                'genres': [c for c in cols if c.startswith('genre_')],
            }
            return df_features

        else:
            raise NotImplementedError(f'Feature method {features_method} not yet implemented')

    # ── Profile Learner ───────────────────────────────────────────────────

    def _build_user_frame(self, u):
        feature_names = list(self.content_features.columns)
        df_user = pd.DataFrame(self.trainset.ur[u], columns=['item_id', 'user_ratings'])
        df_user['item_id'] = df_user['item_id'].map(self.trainset.to_raw_iid)
        df_user = df_user.merge(self.content_features, how='left',
                                left_on='item_id', right_index=True)
        df_user = df_user.dropna(subset=feature_names, how='all').fillna(0)
        if len(df_user) < 2:
            return None, None
        return df_user[feature_names].values, df_user['user_ratings'].values

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        self.user_profile = {u: None for u in trainset.all_users()}

        self.global_mean = trainset.global_mean
        self.user_means = {
            u: np.mean([r for _, r in trainset.ur[u]])
            for u in trainset.all_users()
        }
        self.user_n_ratings = {
            u: len(trainset.ur[u])
            for u in trainset.all_users()
        }

        knn_methods = ('content_knn_centered', 'ridge_knn_blend')
        if self.regressor_method in knn_methods and self.content_features is not None:
            feat_values = self.content_features.values.astype(np.float32)
            self.content_features_norm = pd.DataFrame(
                self._l2_normalize_rows(feat_values),
                index=self.content_features.index,
                columns=self.content_features.columns
            )
            self.user_knn_cache = {}
            for u in trainset.all_users():
                raw_items = [self.trainset.to_raw_iid(iid) for iid, _ in trainset.ur[u]]
                ratings = np.array([r for _, r in trainset.ur[u]], dtype=np.float32)
                feat_rows = self.content_features_norm.reindex(raw_items)
                mask = feat_rows.notna().any(axis=1).values & (
                    np.linalg.norm(feat_rows.fillna(0).values, axis=1) > 0
                )
                if mask.sum() < 2:
                    self.user_knn_cache[u] = None
                    continue
                self.user_knn_cache[u] = (
                    feat_rows.fillna(0).values[mask].astype(np.float32),
                    ratings[mask]
                )

        if self.regressor_method == 'random_score':
            pass

        elif self.regressor_method == 'random_sample':
            for u in self.user_profile:
                self.user_profile[u] = [rating for _, rating in self.trainset.ur[u]]

        elif self.regressor_method == 'stacking_groups':
            assert self.feature_groups is not None, \
                "stacking_groups requires features_method='all_content_v2'"
            self.user_group_models = {}
            self.user_meta_models = {}
            kf = KFold(n_splits=3, shuffle=True, random_state=0)
            group_alphas = [0.1, 1.0, 10.0, 100.0, 1000.0]

            for u in self.user_profile:
                X, y = self._build_user_frame(u)
                if X is None or len(y) < 5:
                    self.user_profile[u] = None
                    continue

                col_idx = {g: [list(self.content_features.columns).index(c) for c in cols]
                           for g, cols in self.feature_groups.items()}

                oof_preds = {g: np.zeros(len(y)) for g in col_idx}
                for tr, va in kf.split(X):
                    for g, idx in col_idx.items():
                        if not idx:
                            continue
                        reg = RidgeCV(alphas=group_alphas, scoring='neg_root_mean_squared_error')
                        reg.fit(X[tr][:, idx], y[tr])
                        oof_preds[g][va] = reg.predict(X[va][:, idx])

                meta_X = np.column_stack([oof_preds[g] for g in col_idx])
                meta = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0],
                               scoring='neg_root_mean_squared_error',
                               fit_intercept=True)
                meta.fit(meta_X, y)
                self.user_meta_models[u] = (meta, list(col_idx.keys()))

                full_models = {}
                for g, idx in col_idx.items():
                    if not idx:
                        continue
                    reg = RidgeCV(alphas=group_alphas, scoring='neg_root_mean_squared_error')
                    reg.fit(X[:, idx], y)
                    full_models[g] = (reg, idx)
                self.user_group_models[u] = full_models
                self.user_profile[u] = True

        elif self.regressor_method == 'elastic_cv':
            for u in self.user_profile:
                X, y = self._build_user_frame(u)
                if X is None:
                    self.user_profile[u] = None
                    continue
                regressor = ElasticNetCV(
                    l1_ratio=[0.1, 0.5, 0.9],
                    alphas=[0.001, 0.01, 0.1, 1.0],
                    cv=3,
                    max_iter=2000,
                    random_state=0,
                    n_jobs=1
                )
                try:
                    regressor.fit(X, y)
                    self.user_profile[u] = regressor
                except Exception:
                    self.user_profile[u] = None

        elif self.regressor_method in (
            'linear_regression', 'random_forest', 'ridge', 'ridge_cv', 'ridge_cv_bias',
            'ridge_cv_centered', 'ridge_knn_blend'
        ):
            for u in self.user_profile:
                X, y = self._build_user_frame(u)
                if X is None:
                    self.user_profile[u] = None
                    continue

                if self.regressor_method == 'ridge_cv_centered':
                    y = y - self.user_means[u]

                if self.regressor_method == 'linear_regression':
                    regressor = LinearRegression(fit_intercept=True)
                elif self.regressor_method == 'random_forest':
                    regressor = RandomForestRegressor(n_estimators=10, random_state=0)
                elif self.regressor_method == 'ridge':
                    regressor = Ridge(alpha=1.0)
                else:
                    regressor = RidgeCV(
                        alphas=[0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0],
                        scoring='neg_root_mean_squared_error'
                    )
                regressor.fit(X, y)
                self.user_profile[u] = regressor

    # ── Scoring component ─────────────────────────────────────────────────

    def _predict_ridge(self, u, raw_item_id):
        if self.user_profile.get(u) is None:
            return self.user_means.get(u, self.global_mean)
        if raw_item_id not in self.content_features.index:
            return self.user_means.get(u, self.global_mean)
        x = self.content_features.loc[[raw_item_id], :].values
        return float(self.user_profile[u].predict(x)[0])

    def _predict_knn(self, u, raw_item_id):
        cache = self.user_knn_cache.get(u)
        if cache is None or raw_item_id not in self.content_features_norm.index:
            return self.user_means.get(u, self.global_mean)
        x_i = self.content_features_norm.loc[raw_item_id].values.astype(np.float32)
        if np.linalg.norm(x_i) == 0:
            return self.user_means.get(u, self.global_mean)
        feat, ratings = cache
        sim = np.clip(feat @ x_i, 0.0, None)
        if sim.sum() <= 1e-9:
            return self.user_means.get(u, self.global_mean)
        k = min(self.knn_k, len(sim))
        if k < len(sim):
            top_idx = np.argpartition(-sim, k - 1)[:k]
            sim = sim[top_idx]
            ratings_used = ratings[top_idx]
        else:
            ratings_used = ratings
        weight_sum = sim.sum()
        if weight_sum <= 1e-9:
            return self.user_means.get(u, self.global_mean)
        user_mean = self.user_means[u]
        return float(user_mean + (sim * (ratings_used - user_mean)).sum() / weight_sum)

    def _predict_stacking(self, u, raw_item_id):
        if self.user_profile.get(u) is None:
            return self.user_means.get(u, self.global_mean)
        if raw_item_id not in self.content_features.index:
            return self.user_means.get(u, self.global_mean)
        x = self.content_features.loc[[raw_item_id], :].values
        group_models = self.user_group_models[u]
        meta, group_order = self.user_meta_models[u]
        sub_preds = []
        for g in group_order:
            reg, idx = group_models[g]
            sub_preds.append(reg.predict(x[:, idx])[0])
        return float(meta.predict(np.array(sub_preds).reshape(1, -1))[0])

    def estimate(self, u, i):
        if not (self.trainset.knows_user(u) and self.trainset.knows_item(i)):
            raise PredictionImpossible('User and/or item is unkown.')

        if self.regressor_method == 'random_score':
            rd.seed()
            return rd.uniform(0.5, 5)
        elif self.regressor_method == 'random_sample':
            rd.seed()
            return rd.choice(self.user_profile[u])

        raw_item_id = self.trainset.to_raw_iid(i)
        lo, hi = self.trainset.rating_scale

        if self.regressor_method in (
            'linear_regression', 'random_forest', 'ridge', 'ridge_cv', 'ridge_cv_bias',
            'ridge_cv_centered', 'elastic_cv'
        ):
            if self.content_features is None or raw_item_id not in self.content_features.index:
                return float(np.clip(self.user_means.get(u, self.global_mean), lo, hi))

            score = self._predict_ridge(u, raw_item_id)

            if self.regressor_method == 'ridge_cv_bias':
                n_ratings = self.user_n_ratings[u]
                weight = min(1.0, n_ratings / 50)
                score = weight * score + (1 - weight) * self.user_means[u]
            elif self.regressor_method == 'ridge_cv_centered':
                score = score + self.user_means[u]

            return float(np.clip(score, lo, hi))

        elif self.regressor_method == 'content_knn_centered':
            return float(np.clip(self._predict_knn(u, raw_item_id), lo, hi))

        elif self.regressor_method == 'ridge_knn_blend':
            return float(np.clip(
                0.5 * self._predict_ridge(u, raw_item_id)
                + 0.5 * self._predict_knn(u, raw_item_id),
                lo, hi
            ))

        elif self.regressor_method == 'stacking_groups':
            return float(np.clip(self._predict_stacking(u, raw_item_id), lo, hi))

        return self.global_mean

The following script test the ContentBased class

In [25]:
def test_contentbased_class(feature_method, regressor_method):
    """Test the ContentBased class.
    Tries to make a prediction on the first (user,item) tuple of the anti_test_set
    """
    sp_ratings = load_ratings(surprise_format=True)
    train_set = sp_ratings.build_full_trainset()
    content_algo = ContentBased(feature_method, regressor_method)
    content_algo.fit(train_set)
    anti_test_set_first = train_set.build_anti_testset()[0]
    prediction = content_algo.predict(anti_test_set_first[0], anti_test_set_first[1])
    print(prediction)

# Test 1 : random score (no features)
print("=== Test with random_score ===")
test_contentbased_class(None, 'random_score')

# Test 2 : random sample (no features)
print("=== Test with random_sample ===")
test_contentbased_class(None, 'random_sample')

# Test 3 : linear regression (title_length feature)
print("=== Test with linear_regression ===")
test_contentbased_class("title_length", "linear_regression")

=== Test with random_score ===
user: 277        item: 3          r_ui = None   est = 1.33   {'was_impossible': False}
=== Test with random_sample ===
user: 277        item: 3          r_ui = None   est = 5.00   {'was_impossible': False}
=== Test with linear_regression ===


NotImplementedError: Feature method title_length not yet implemented